# Aula 08 · JSON, datas e exceções

Este caderno é o [capítulo 8 do site](https://lacouth.github.io/python_telecom-site/unidade3-arquivos/08-json-datas-excecoes/) em forma de
aula: o mesmo texto, os mesmos exemplos, **sem as saídas**. Em cada exemplo:

1. **leia** o código, sem rodar;
2. **escreva** na célula `_Sua previsão:_`, logo abaixo dele, o que você acha
   que vai sair;
3. **rode** a célula do código e compare com o que você escreveu;
4. **abra** o `▶ O que aconteceu` para ler a explicação.

A previsão errada é a parte que ensina — não a apague.

**Ao fim desta aula você deve conseguir:**

1. converter JSON em estrutura Python e de volta, e reconhecer o que torna um JSON inválido;
2. calcular duração entre dois timestamps e explicar por que comparar datas como texto não serve;
3. proteger o processamento de um item com `try/except` específico, registrando o descarte.

## Parte 1 — O capítulo, exemplo a exemplo

Três assuntos que parecem independentes e que, juntos, são o que separa um script
de exercício de um coletor que roda sozinho todo dia: o formato em que os sistemas
conversam (**JSON**), a grandeza que quase todo indicador de operação usa
(**tempo**) e a defesa contra o dado estranho que sempre chega (**exceções**).

## JSON

In [ ]:
import json

texto = '''
{
  "coleta": "2026-03-02T23:59:00",
  "equipamentos": [
    {"nome": "OLT-CENTRO-01", "tipo": "OLT", "portas": 16, "em_servico": true},
    {"nome": "ONU-SUL-4512", "tipo": "ONU", "portas": 1, "em_servico": false}
  ]
}
'''

# json.loads le de uma STRING; json.load le de um arquivo aberto.
dados = json.loads(texto)

print(type(dados))
print(dados["coleta"])
print(len(dados["equipamentos"]))
print(dados["equipamentos"][0]["nome"])
print(dados["equipamentos"][1]["em_servico"])

# O caminho de volta: de estrutura Python para texto JSON.
resumo = {"total": 2, "em_servico": 1}
print(json.dumps(resumo))
print(json.dumps(resumo, indent=2, ensure_ascii=False))


**Preveja:** "que tipo o `loads` devolve?"

_Sua previsão:_

→ escreva aqui e, só então, rode a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

JSON é um formato de texto para estruturas de dados, e a correspondência com
Python é quase direta:

| JSON | Python |
|---|---|
| objeto `{ }` | `dict` |
| lista `[ ]` | `list` |
| string | `str` (sempre com **aspas duplas**) |
| número | `int` ou `float` |
| `true` / `false` | `True` / `False` |
| `null` | `None` |

- **`json.loads(texto)`** lê de uma string; **`json.load(arquivo)`** lê de um
  arquivo aberto (o `s` é de *string*);
- **`json.dumps(estrutura)`** faz o caminho inverso. Com `indent=2` sai legível
  para humano; com `ensure_ascii=False`, os acentos saem como acentos, e não como
  `ç`.

Depois do `loads`, você tem dicionários e listas — nada de novo. `dados["equipamentos"][0]["nome"]`
é a mesma navegação do capítulo 6.

Duas diferenças em relação a Python que derrubam quem escreve JSON à mão:

</details>

In [ ]:
import json
# Aspas simples nao sao JSON valido -- JSON so aceita aspas duplas.
json.loads("{'nome': 'OLT-CENTRO-01'}")


**Preveja:** "o que há de errado neste JSON?"

_Sua previsão:_

→ escreva aqui e, só então, rode a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

JSON só aceita **aspas duplas**, e os booleanos são `true`/`false` em minúsculas.

</details>

## Datas

In [ ]:
from datetime import datetime, timedelta

# strptime le uma data escrita em texto, seguindo o formato que voce descreve.
momento = datetime.strptime("2026-03-02 14:03:17", "%Y-%m-%d %H:%M:%S")
print(momento)
print(momento.year, momento.hour)

# Subtrair duas datas da um timedelta -- uma DURACAO.
inicio = datetime.strptime("2026-03-02 14:03:17", "%Y-%m-%d %H:%M:%S")
fim = datetime.strptime("2026-03-02 15:48:17", "%Y-%m-%d %H:%M:%S")
duracao = fim - inicio
print(duracao)
print(duracao.total_seconds())
print(round(duracao.total_seconds() / 60, 1))

# Comparar datas e comparar numeros.
print(fim > inicio)

# Somar tempo a uma data.
print(inicio + timedelta(hours=2))

# strftime faz o caminho inverso: data para texto, no formato que voce quiser.
print(momento.strftime("%d/%m/%Y"))
print(momento.strftime("%H:%M"))


**Preveja:** "a subtração de duas datas devolve um número ou outra coisa?"

_Sua previsão:_

→ escreva aqui e, só então, rode a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Um timestamp de log é texto. Como texto, ele serve para ler e comparar
alfabeticamente — e só. Para calcular qualquer coisa, converta para `datetime`:

- **`strptime(texto, formato)`** interpreta o texto; o formato descreve o que está
  escrito. Os códigos que você vai usar: `%Y` (ano com 4 dígitos), `%m` (mês),
  `%d` (dia), `%H` (hora), `%M` (minuto), `%S` (segundo);
- **`strftime(formato)`** faz o inverso, para imprimir do jeito que você quiser;
- **subtrair duas datas** dá um `timedelta`, que é uma **duração**;
  `.total_seconds()` a converte para número;
- **somar** um `timedelta` a uma data dá outra data;
- datas se comparam com `<` e `>` como números.

Esse quarteto — ler, subtrair, comparar e formatar — é tudo que se precisa para
calcular uma janela de indisponibilidade, o tempo entre a abertura e o
encerramento de um alarme ou o intervalo coberto por um arquivo de log.

Comparar datas **como texto** às vezes funciona e às vezes não, e é difícil
perceber quando:

</details>

In [ ]:
from datetime import datetime

# 31 de dezembro vem ANTES de 1 de janeiro -- mas nao em ordem alfabetica.
print("31/12/2025" < "01/01/2026")

inicio = datetime.strptime("31/12/2025", "%d/%m/%Y")
fim = datetime.strptime("01/01/2026", "%d/%m/%Y")
print(inicio < fim)


**Preveja:** "31 de dezembro vem antes de 1º de janeiro. As duas linhas concordam?"

_Sua previsão:_

→ escreva aqui e, só então, rode a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

No formato `AAAA-MM-DD` do log, a ordem alfabética coincide com a cronológica — e
só por isso a comparação de texto funciona. Numa data brasileira, não: lido da
esquerda para a direita, o `31` de dezembro parece vir depois do `01` de janeiro.

</details>

In [ ]:
from datetime import datetime
# O formato descrito nao bate com o texto recebido.
datetime.strptime("02/03/2026", "%Y-%m-%d")


**Preveja:** "qual é a mensagem quando o formato não descreve o texto?"

_Sua previsão:_

→ escreva aqui e, só então, rode a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

A mensagem é literal: o texto não bate com o formato descrito. Em geral é uma data
brasileira (`02/03/2026`) sendo lida com o formato ISO — e o formato certo seria
`"%d/%m/%Y"`.

</details>

## Exceções: o coletor que não morre

In [ ]:
linhas = [
    "2026-03-02 14:03:17 CRITICAL OLT-CENTRO-01 perda de sinal",
    "",
    "linha truncada",
    "2026-03-02 10:02:55 CRITICAL ONU-SUL-4512 sem resposta",
]

# Sem defesa nenhuma, a segunda linha derruba o programa inteiro.
# Com try/except POR ITEM, a linha ruim e registrada e o resto continua.
criticos = 0
descartadas = 0

for linha in linhas:
    try:
        if linha.split()[2] == "CRITICAL":
            criticos = criticos + 1
    except IndexError:
        descartadas = descartadas + 1

print(criticos)
print(descartadas)


def para_numero(texto):
    """Converte para float, devolvendo None quando o texto nao e numero."""
    try:
        return round(float(texto), 2)
    except ValueError:
        return None


print(para_numero("-21.456"))
print(para_numero("sem leitura"))
print(para_numero(""))


**Preveja:** "quantos críticos e quantas linhas descartadas?"

_Sua previsão:_

→ escreva aqui e, só então, rode a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`try/except` diz: *tente isto; se der este erro, faça aquilo em vez de parar*.

Repare em **onde** o `try` está: dentro do laço, envolvendo o processamento de
**uma** linha. É a diferença entre perder uma linha e perder o arquivo inteiro.
Envolver o laço todo num `try` faz o programa abortar na primeira linha ruim — que
é exatamente o que se queria evitar.

</details>

In [ ]:
linhas = [
    "2026-03-02 14:03:17 CRITICAL OLT-CENTRO-01 perda de sinal",
    "",
    "linha truncada",
    "2026-03-02 10:02:55 CRITICAL ONU-SUL-4512 sem resposta",
]

# O MESMO codigo, com o try em volta do laco inteiro.
criticos = 0
try:
    for linha in linhas:
        if linha.split()[2] == "CRITICAL":
            criticos = criticos + 1
except IndexError:
    pass
print(criticos)


**Preveja:** "quantos críticos esta versão conta?"

_Sua previsão:_

→ escreva aqui e, só então, rode a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Um só: ao encontrar a segunda linha, o erro sobe, o `except` captura e o laço é
abandonado — o alarme crítico da quarta linha nunca chega a ser visto.

E repare no que o `except` faz: ele **registra** a linha descartada. Um `except`
que não registra nada é pior que erro nenhum, porque transforma "processei 998 de
1000 linhas" em "processei tudo, tá ótimo".

!!! danger "Nunca escreva `except:` sozinho"
    Um `except` sem tipo captura **todos** os erros, inclusive os de digitação no
    seu próprio código — e o defeito passa a ser invisível. Capture o erro que
    você espera: `except ValueError`, `except IndexError`,
    `except FileNotFoundError`. Se não sabe qual é, rode uma vez e leia o nome na
    última linha do traceback.

A segunda metade do exemplo mostra o formato que você vai repetir bastante: uma
função pequena que **tenta converter e devolve `None` quando não dá**. Quem chama
decide o que fazer com o `None` — e é sempre mais fácil decidir isso de fora da
função do que dentro dela.

</details>

### O padrão completo

In [ ]:
from datetime import datetime

LINHAS = [
    "2026-03-02 14:03:17 CRITICAL OLT-CENTRO-01 perda de sinal",
    "data-invalida 99:99:99 CRITICAL OLT-CENTRO-01 relogio errado",
    "2026-03-02 15:48:17 INFO OLT-CENTRO-01 alarme encerrado",
    "",
]


def momento_da_linha(linha):
    """Devolve o datetime da linha, ou None se ela nao estiver no formato."""
    campos = linha.split()
    if len(campos) < 5:
        return None
    try:
        return datetime.strptime(f"{campos[0]} {campos[1]}", "%Y-%m-%d %H:%M:%S")
    except ValueError:
        return None


momentos = []
problemas = 0
for linha in LINHAS:
    momento = momento_da_linha(linha)
    if momento is None:
        problemas = problemas + 1
        continue
    momentos.append(momento)

print(len(momentos), problemas)
print(min(momentos))
print(max(momentos))
print((max(momentos) - min(momentos)).total_seconds() / 60)


**Preveja:** "o programa informa quantas linhas processou e quantas descartou. Por que as duas?"

_Sua previsão:_

→ escreva aqui e, só então, rode a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Este é o formato que o projeto final vai usar, e vale ler com atenção. A função
`momento_da_linha` faz três coisas nesta ordem:

1. **verifica o que dá para verificar antes** (`len(campos) < 5`) — é mais barato e
   mais claro que deixar dar erro;
2. **protege com `try/except` o que não dá para verificar antes** — não existe jeito
   razoável de saber se um texto é uma data válida sem tentar convertê-lo;
3. **devolve `None` em vez de levantar**, deixando a decisão para quem chamou.

E quem chama conta os problemas separadamente, em vez de fingir que não houve. No
fim, o programa entrega dois números: **o que processou e o que descartou**. Sem o
segundo, o primeiro não vale nada.

</details>

## Onde isso é usado de verdade

JSON é a língua franca entre sistemas de rede. A API do
[NetBox](https://netboxlabs.com/docs/netbox/) — hoje o padrão de *source of truth*
de inventário — responde em JSON; os modelos OpenConfig/YANG, que padronizam a
configuração entre fabricantes, são serializados em JSON; e as
[ferramentas de automação](https://blog.apnic.net/2023/02/13/automation-tools-paramiko-netmiko-napalm-ansible-nornir-or/)
trocam estado nesse formato. Quando você consultar uma API na Unidade 7, o que vai
chegar na sua variável é exatamente o que este capítulo mostrou.

As contas com data são os indicadores de operação: **MTTR** (tempo médio de
reparo), duração de cada janela de indisponibilidade, aderência ao SLA, correlação
entre alarmes que aconteceram "quase ao mesmo tempo". Todos saem de subtrair dois
timestamps.

E o tratamento de erro é o que torna um coletor utilizável. Rodando sobre um
inventário de centenas de equipamentos, **sempre** haverá um que não responde, um
que responde num formato inesperado e um com o relógio errado. Um coletor que
aborta no primeiro problema não coleta nada; o que ignora os problemas em silêncio
é pior, porque entrega um relatório incompleto com cara de completo.

## Erros comuns deste capítulo

| Sintoma | Causa provável |
|---|---|
| `JSONDecodeError` | aspas simples, vírgula sobrando ou `True` em vez de `true` |
| `ValueError: time data ... does not match format` | o formato do `strptime` não descreve o texto |
| `TypeError` ao somar datas | não se somam duas datas; soma-se `timedelta` a uma data |
| o `except` não pegou o erro | você capturou o tipo errado; leia a última linha do traceback |
| o programa "funciona" mas o total está baixo | o `except` engoliu linhas sem registrar |
| tudo parou na primeira linha ruim | o `try` está em volta do laço, e não dentro dele |
| `AttributeError: 'str' object has no attribute 'year'` | o timestamp ainda é texto; faltou o `strptime` |

## Resumo

- JSON vira dicionário e lista; `loads`/`dumps` para string, `load`/`dump` para
  arquivo.
- JSON só tem aspas duplas, e `true`/`false` em minúsculas.
- `strptime` lê data de texto, `strftime` escreve; a diferença entre duas datas é
  um `timedelta`.
- `try/except` **por item**, dentro do laço.
- Capture o erro específico; `except:` sozinho esconde defeito.
- Sempre **registre** o que foi descartado, e informe quantos foram.

_Anotações da Parte 1:_

## Parte 2 — Resolver junto

Tente sozinho primeiro, por cinco minutos, na sua máquina. Depois
resolvemos juntos.

### E1. Ler o JSON da coleta

A resposta de um sistema de gerência chega como texto JSON:

```json
{"coleta": "2026-03-02", "equipamentos": [{"nome": "OLT-A", "portas": 16}]}
```

Escreva `equipamentos(texto)`, que converte o texto e devolve a **lista** de
equipamentos.

```python
equipamentos('{"coleta": "2026-03-02", "equipamentos": [{"nome": "OLT-A"}]}')
# -> [{"nome": "OLT-A"}]
```

Se a chave `"equipamentos"` não existir, devolva lista vazia.

**Assinatura:**

```python
import json

def equipamentos(texto):
    """Lista de equipamentos contida no JSON recebido."""
```

In [ ]:
# sua solução aqui


### E2. Momento e duração

Escreva `momento(texto)`, que converte um timestamp no formato
`"AAAA-MM-DD HH:MM:SS"` para `datetime`. Se o texto não estiver nesse formato,
devolva `None` — **sem deixar o erro subir**.

```python
momento("2026-03-02 14:03:17")   # -> datetime(2026, 3, 2, 14, 3, 17)
momento("02/03/2026")            # -> None
momento("")                      # -> None
```

**Assinatura:**

```python
from datetime import datetime

def momento(texto):
    """Converte o timestamp para datetime, ou None se o formato não bater."""
```

Escreva `duracao_minutos(inicio, fim)`, que recebe dois timestamps em texto e
devolve a duração em **minutos**, com uma casa decimal.

```python
duracao_minutos("2026-03-02 14:03:17", "2026-03-02 15:48:17")   # -> 105.0
```

Se qualquer um dos dois não estiver no formato esperado, devolva `None`.

**Assinatura:**

```python
from datetime import datetime

def duracao_minutos(inicio, fim):
    """Minutos entre os dois timestamps, ou None se algum for inválido."""
```

In [ ]:
# sua solução aqui


### E3. A média que informa o descarte

Escreva `media_valida(textos)`, que recebe uma lista de leituras em texto e
devolve a tupla `(media, descartadas)`: a média dos valores que dava para
converter (duas casas) e **quantos** foram descartados.

```python
media_valida(["-21.4", "sem sinal", "-19.8"])   # -> (-20.6, 1)
media_valida(["erro", "erro"])                  # -> (0.0, 2)
media_valida([])                                # -> (0.0, 0)
```

Este é o princípio da unidade: o programa não morre por causa de uma leitura ruim,
mas também **não finge que ela não existiu**.

**Assinatura:**

```python
def media_valida(textos):
    """Devolve (média das leituras válidas, quantidade de descartadas)."""
```

In [ ]:
# sua solução aqui


### E4. O coletor completo

Feche a unidade com o coletor completo. Escreva `processa_log(linhas)`, que
devolve um **dicionário** com o resumo do log:

```python
{
    "processadas": 3,        # linhas em formato válido
    "ignoradas": 2,          # linhas descartadas
    "criticos": 2,           # alarmes CRITICAL entre as processadas
    "equipamentos": 2,       # equipamentos distintos que apareceram
    "minutos": 105.0,        # intervalo coberto pelo log
}
```

Para um log sem nenhuma linha válida, todos os números são `0` (e `minutos` é
`0.0`).

É o mesmo relatório que o projeto final vai produzir — e repare que ele informa
**as duas coisas**: o que foi processado e o que foi descartado.

**Assinatura:**

```python
from datetime import datetime

FORMATO = "%Y-%m-%d %H:%M:%S"

def momento_da_linha(campos):
    """Devolve o datetime dos dois primeiros campos, ou None. JÁ ESCRITA."""
    try:
        return datetime.strptime(f"{campos[0]} {campos[1]}", FORMATO)
    except ValueError:
        return None

def processa_log(linhas):
    """Resumo do log: processadas, ignoradas, críticos, equipamentos e minutos."""
    #   1. separe os campos; se forem menos de 5, é ignorada -> continue
    #   2. chame momento_da_linha(campos); se devolver None, é ignorada -> continue
    #   3. senão, conte como processada, guarde o momento numa lista, guarde o
    #      equipamento (campos[3]) num conjunto, e conte se campos[2] é CRITICAL
    # No fim, os minutos saem de max(momentos) - min(momentos).
```

In [ ]:
# sua solução aqui


_Anotações da Parte 2:_

## Parte 3 — Quiz de conceitos

Responda de cabeça, sem rodar. Conferimos juntos no fim.

**1.** `json.loads('{"portas": 16}')` devolve:

a) uma string  b) um dicionário  c) uma lista  d) um objeto JSON

**2.** É JSON válido:

a) `{'nome': 'OLT-A'}`  b) `{"ativo": True}`  c) `{"ativo": true}`  d) `{nome: "OLT-A"}`

**3.** Subtrair dois `datetime` devolve:

a) um número de segundos  b) um `timedelta`  c) uma string  d) erro

**4.** O `try/except` que envolve o **laço inteiro**, num arquivo cuja terceira linha está malformada:

a) processa tudo  b) processa até a segunda linha e para  c) pula só a terceira  d) não processa nada

**5.** `except: pass` é má prática porque:

a) é mais lento  b) esconde qualquer erro, inclusive os do seu próprio código  c) não compila  d) só funciona com `ValueError`